In [1]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)

import os
import re
import requests
import pandas as pd
import OpenDartReader

pd.options.display.max_colwidth = None
pd.options.display.max_rows = None

def make_clickable(val):
    return f'<a href="{val}" target="_blank">{val}</a>'

api_key = '888e4aca3aba61a62811ba87e0fc054e2d9f6ea0'
dart = OpenDartReader(api_key)

corp_code = '00126380'

# 1. 사업보고서 목록 조회
reports = dart.list(corp_code, start='2026-01-01', end='2026-12-31', kind='A', final=True)
display(reports)

,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,00126380,삼성전자,005930,Y,사업보고서 (2025.12),20260310002820,삼성전자,20260310,연


In [4]:
rcp_no = '20260310002820'
df = dart.sub_docs(rcp_no)
display(df.style.format({'url': make_clickable}))

,title,url
0,사 업 보 고 서,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=1&offset=957&length=4524&dtd=dart4.xsd
1,【 대표이사 등의 확인 】,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=2&offset=5585&length=451&dtd=dart4.xsd
2,I. 회사의 개요,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=3&offset=6040&length=197163&dtd=dart4.xsd
3,1. 회사의 개요,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=4&offset=6176&length=45026&dtd=dart4.xsd
4,2. 회사의 연혁,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=5&offset=51206&length=25144&dtd=dart4.xsd
5,3. 자본금 변동사항,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=6&offset=76354&length=3406&dtd=dart4.xsd
6,4. 주식의 총수 등,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=7&offset=79764&length=109711&dtd=dart4.xsd
7,5. 정관에 관한 사항,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=8&offset=189479&length=13710&dtd=dart4.xsd
8,II. 사업의 내용,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=9&offset=203207&length=151898&dtd=dart4.xsd
9,1. 사업의 개요,http://dart.fss.or.kr/report/viewer.do?rcpNo=20260310002820&dcmNo=11104488&eleId=10&offset=203605&length=2305&dtd=dart4.xsd


In [6]:
save_dir = f'html_reports/{rcp_no}'
os.makedirs(save_dir, exist_ok=True)

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0'
})

for idx, row in df.iterrows():
    title = row['title']
    url = row['url']

    safe_title = re.sub(r'[\\/:*?"<>|]+', '_', title).strip()
    file_name = f'{idx:02d}_{safe_title}.html'
    save_path = os.path.join(save_dir, file_name)

    resp = session.get(url, timeout=30)
    resp.raise_for_status()

    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(resp.text)

    print(f'Saved: {save_path}')

Saved: html_reports/20260310002820/00_사 업 보 고 서.html
Saved: html_reports/20260310002820/01_【 대표이사 등의 확인 】.html
Saved: html_reports/20260310002820/02_I. 회사의 개요.html
Saved: html_reports/20260310002820/03_1. 회사의 개요.html
Saved: html_reports/20260310002820/04_2. 회사의 연혁.html
Saved: html_reports/20260310002820/05_3. 자본금 변동사항.html
Saved: html_reports/20260310002820/06_4. 주식의 총수 등.html
Saved: html_reports/20260310002820/07_5. 정관에 관한 사항.html
Saved: html_reports/20260310002820/08_II. 사업의 내용.html
Saved: html_reports/20260310002820/09_1. 사업의 개요.html
Saved: html_reports/20260310002820/10_2. 주요 제품 및 서비스.html
Saved: html_reports/20260310002820/11_3. 원재료 및 생산설비.html
Saved: html_reports/20260310002820/12_4. 매출 및 수주상황.html
Saved: html_reports/20260310002820/13_5. 위험관리 및 파생거래.html
Saved: html_reports/20260310002820/14_6. 주요계약 및 연구개발활동.html
Saved: html_reports/20260310002820/15_7. 기타 참고사항.html
Saved: html_reports/20260310002820/16_III. 재무에 관한 사항.html
Saved: html_reports/20260310002820/17_1. 요약재무정보.html
Sav

## 감사보고서 PDF 일괄 다운로드 기능\n
이 셀을 먼저 실행하여 함수를 정의하세요.

In [ ]:
import os

def download_audit_reports(corp_code, start_date):
    print(f"Searching for reports for {corp_code} since {start_date}...")
    # 1. 보고서 목록 조회 (F: 정기공시 - 감사보고서 포함)
    reports = dart.list(corp_code, start=start_date, kind='F')
    
    if reports is None or len(reports) == 0:
        print("No reports found.")
        return

    # '감사보고서' 또는 '연결감사보고서' 키워드가 포함된 보고서만 필터링
    audit_reports = reports[reports['report_nm'].str.contains('감사보고서')]
    
    if len(audit_reports) == 0:
        print("No audit reports found in the search results.")
        return
    
    print(f"Found {len(audit_reports)} audit report(s).")

    if not os.path.exists('PDF'):
        os.makedirs('PDF')
        print("Created 'PDF' directory.")
        
    downloaded_count = 0
    for _, row in audit_reports.iterrows():
        rcp_no = row['rcept_no']
        report_nm = row['report_nm']
        print(f"\nChecking attachments for: {report_nm} ({rcp_no})")
        
        try:
            files = dart.attach_files(rcp_no)
            if not files:
                print("  No attachments found.")
                continue
                
            for filename, url in files.items():
                if filename.lower().endswith('.pdf'):
                    # 파일명에 특수문자 제거 및 경로 설정
                    clean_filename = filename.replace('/', '_').replace('\\', '_')
                    save_path = os.path.join('PDF', clean_filename)
                    
                    print(f"  Downloading: {filename}")
                    dart.download(url, save_path)
                    downloaded_count += 1
        except Exception as e:
            print(f"  Error accessing attachments: {e}")
                
    print(f"\nFinished! Successfully downloaded {downloaded_count} PDF(s) to 'PDF/' folder.")

### 다운로드 실행\n
아래 셀을 실행하면 실제 다운로드가 시작됩니다.

In [15]:
# 구다이글로벌('01753163')의 2024년 이후 모든 감사보고서 PDF 다운로드 실행
download_audit_reports('01753163', '2024-01-01')


Checking report: [기재정정]감사보고서 (2023.12) (20251118000238)
  Downloading: [구다이글로벌][정정]감사보고서(2025.11.18).pdf

Checking report: 연결감사보고서 (2024.12) (20250430000504)
  Downloading: [구다이글로벌]연결감사보고서(2025.04.30).pdf

Checking report: 감사보고서 (2024.12) (20250411000394)
  Downloading: [구다이글로벌]감사보고서(2025.04.11).pdf

Finished! Downloaded 3 PDF(s).
